# Baseline notebook

Create model baseline and analysis for it

In [ ]:
import pandas as pd 
import numpy as np

pd.set_option('display.max_columns', 200)
np.random.seed(42)

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path)
df.head()

## Data preparation (From EDA)

In [ ]:
to_category_columns = (
    [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "Churn"
        ])
    
for column in to_category_columns:
    df[column] = df[column].astype("category")
    
df["SeniorCitizen"] = df["SeniorCitizen"] == 1
#FIXME: take a look later
df["Churn"] = df["Churn"] == "Yes"
    
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')

df = df.dropna()
df["TotalCharges"].isna().sum()

print("Data preparation is done")
df.shape

## Model building

In [ ]:
df.columns

In [ ]:
# Drop customerId column as never significant
df = df.drop(columns="customerID")
df.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

df_ohe = pd.get_dummies(df)
df_ohe.shape

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

rf = RandomForestClassifier(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(rf, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

### Metrics explained

It simpler to explain on one class. I will use where Churn is True

Class True (Churn is True):
- Precision 0.64 — when model predicts "churn", it's correct only 64% of the time
- Recall 0.48 — it catches only 48% of actual churners

Same logic for the not churned users

As we are trying to minimize missed by model churns (FN error that costs business big money). We should maximize recall on True class.

Despite model performs well on false classes (can classify not churning customers) it's perform nearly a coin flip predict on actually churned ones. Such conduct can be caused by class imbalance (roughly $\approx30%$% customers from data were churned)

## Data imbalance handling

Let's evaluate class imbalance handling methods

In [ ]:
churn_ratio = df_ohe.groupby("Churn")["Churn"].count()
churn_ratio

In [ ]:
print(f"Churned to not churned ratio is {churn_ratio.iloc[1] / churn_ratio.iloc[0]:.2f}")

### Model tuning (class_weight = 'balanced')

In [ ]:
# Simply change parameter in model
rf = RandomForestClassifier(random_state=42, class_weight="balanced")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(rf, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

Metrics do not change at all. Class weights do not bring significant result

### Using GBDT model

As GBDT models are robust to class imbalance I will perform analysis also

I will train a LightGBM as GBDT model with simple `is_unbalanced` parameter for handling data imbalance

In [ ]:
%pip install lightgbm -q

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(is_unbalance=True, verbose=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(lgbm, X, y, cv=skf)
print(classification_report(y, y_pred_cv))

GBDT with native data imbalance handling parameter performs much better than random forest. We gain 75% recall on our churned customers that is a good result for almost not tuned model

### Undersampling

In [ ]:
minority = df_ohe[df["Churn"] == True]
majority = df_ohe[df["Churn"] == False]

print(f"minority samples: {minority.shape[0]}")
print(f"majority samples: {majority.shape[0]}")

In [ ]:
majority = majority.sample(n=minority.shape[0], replace=False, random_state=42)

undersampled_df = pd.concat([minority, majority]).reset_index(drop=True)
churn_ratio = undersampled_df.groupby("Churn")["Churn"].count()
churn_ratio

In [ ]:
rf = RandomForestClassifier(random_state=42)

#skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_und = undersampled_df.drop(columns="Churn").copy()
y_und = undersampled_df["Churn"].copy()

y_pred_cv = cross_val_predict(rf, X_und, y_und, cv=5)
print(classification_report(y_und, y_pred_cv))

Undersampled metrics are quite impressive. We achieve score like a GBDT model but using baseline. Undersample have decrease metrics on False classes (Model do more FP mistakes, thinking that customer will churn, while he will not). For our case this is acceptable (we are focusing on detecting True classes)